# Model Ensemble Lab

Use this notebook to compare the generated submissions and build custom chapter/category ensembles.

The notebook does not retrain the slow models by default. It works from the CSV outputs already generated in `output/` and the previous scored submission in `comparisson/`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / 'output'
SUBMISSIONS_DIR = OUTPUT_DIR / 'submissions'
PREDICTIONS_DIR = OUTPUT_DIR / 'predictions'
COMPARISON_DIR = PROJECT_ROOT / 'comparisson'

print(PROJECT_ROOT)

## Load Submissions

For category scoring, full-code submissions are converted to category by taking the first character of `Code`.

In [ ]:
def load_category_submission(path, name):
    df = pd.read_csv(path).sort_values('id').reset_index(drop=True)
    out = df[['id']].copy()
    if 'y_category' in df.columns:
        out[name] = df['y_category'].astype(str)
    elif 'Code' in df.columns:
        out[name] = df['Code'].astype(str).str[0]
    else:
        raise ValueError(f'No y_category or Code column in {path}')
    return out

sources = {
    'old_0567': COMPARISON_DIR / 'final_submission.csv',
    'chapter': SUBMISSIONS_DIR / 'kaggle_submission_chapter.csv',
    'full_basic_as_chapter': SUBMISSIONS_DIR / 'kaggle_submission.csv',
    'full_optimized_as_chapter': SUBMISSIONS_DIR / 'kaggle_submission_optimized.csv',
    'full_meta_as_chapter': SUBMISSIONS_DIR / 'kaggle_submission_meta_ensemble.csv',
}

preds = None
for name, path in sources.items():
    if path.exists():
        one = load_category_submission(path, name)
        preds = one if preds is None else preds.merge(one, on='id', how='inner')
        print(f'loaded {name}: {path.name}')
    else:
        print(f'missing {name}: {path}')

preds.head()

## Agreement Matrix

This shows how similar the prediction files are to each other. Higher agreement means two submissions will likely score similarly.

In [ ]:
model_cols = [c for c in preds.columns if c != 'id']
agreement = pd.DataFrame(index=model_cols, columns=model_cols, dtype=float)

for a in model_cols:
    for b in model_cols:
        agreement.loc[a, b] = (preds[a] == preds[b]).mean()

agreement.round(4)

## Load Model Confidence

The chapter model includes a `chapter_confidence` margin. Higher values were much more reliable in validation.

In [ ]:
chapter_detail = pd.read_csv(PREDICTIONS_DIR / 'final_chapter_predictions.csv').sort_values('id').reset_index(drop=True)
chapter_detail[['id', 'Literal', 'y_category', 'chapter_confidence']].head()

## Custom Weighted Vote

Edit the weights below. The vote is over category predictions, not full ICD codes.

In [ ]:
weights = {
    'old_0567': 1.0,
    'chapter': 1.0,
    'full_basic_as_chapter': 0.4,
    'full_optimized_as_chapter': 0.5,
    'full_meta_as_chapter': 0.6,
}

def weighted_vote(row, weights):
    scores = {}
    for col, weight in weights.items():
        if col not in row or pd.isna(row[col]):
            continue
        scores[row[col]] = scores.get(row[col], 0.0) + weight
    return max(scores.items(), key=lambda item: item[1])[0]

custom = preds[['id']].copy()
custom['Literal'] = pd.read_csv(COMPARISON_DIR / 'final_submission.csv').sort_values('id')['Literal'].values
custom['y_category'] = preds.apply(lambda row: weighted_vote(row, weights), axis=1)

print(custom['y_category'].value_counts().head(15))
custom.head()

## Confidence Hybrid

This starts with the previous `.567` submission and only overrides rows where our chapter model is confident.

In [ ]:
threshold = 0.278  # try 0.0, 0.278, 0.5, 0.75, 1.0

old = pd.read_csv(COMPARISON_DIR / 'final_submission.csv').sort_values('id').reset_index(drop=True)
hybrid = old.copy()
use_ours = chapter_detail['chapter_confidence'] >= threshold
hybrid.loc[use_ours, 'y_category'] = chapter_detail.loc[use_ours, 'y_category'].values

changed = (old['y_category'].astype(str) != hybrid['y_category'].astype(str))
print(f'override rows: {use_ours.sum()}')
print(f'changed rows:  {changed.sum()}')
print(f'agreement with old: {(~changed).mean():.4f}')
hybrid.head()

## Save A Submission

Choose which dataframe to save: `custom` for weighted vote, or `hybrid` for confidence override.

In [ ]:
submission_to_save = hybrid  # change to custom if you want the weighted vote
output_name = 'kaggle_submission_notebook_custom.csv'

path = SUBMISSIONS_DIR / output_name
submission_to_save.to_csv(path, index=False)
print(path)
print(submission_to_save.shape)
print(list(submission_to_save.columns))

## Compare Two Submissions

Use this to inspect what changed before uploading to Kaggle.

In [ ]:
a = old.copy()
b = submission_to_save.copy()

same = a['y_category'].astype(str) == b['y_category'].astype(str)
diff = pd.DataFrame({
    'id': a.loc[~same, 'id'],
    'Literal': a.loc[~same, 'Literal'],
    'old': a.loc[~same, 'y_category'].astype(str),
    'new': b.loc[~same, 'y_category'].astype(str),
})

print(f'same: {same.sum()}')
print(f'changed: {(~same).sum()}')
display(diff.head(25))
display(diff.value_counts(['old', 'new']).head(20))